# Capability: regex_extraction

The `regex_extraction` capability uses regular expressions to extract field values. When a contract field specifies a `pattern`, the planner assigns `regex_extraction` as the primary extraction strategy.

This is ideal for fields with well-defined formats: invoice numbers, order IDs, dates, phone numbers, etc.

In [ ]:
from pydantic import BaseModel, Field

import paxman
import paxman.contract.adapters.pydantic


class Order(BaseModel):
    invoice_number: str = Field(..., pattern=r"INV-\d{4}-\d{4}")
    date: str = Field(..., pattern=r"\d{4}-\d{2}-\d{2}")
    supplier: str


sample = """
Invoice INV-2026-0042
Date: 2026-06-30
Supplier: ACME Hardware Supplies
"""
result = paxman.normalize(sample, Order)
print(f"Data: {result.normalized_data}")

In [ ]:
# Show the evidence with line-level provenance
for field_name, ev in result.evidence.items():
    print(f"--- {field_name} ---")
    for candidate in ev.evidence:
        print(f"  Value: {candidate.value!r}")
        print(f"  Confidence: {candidate.confidence.name}")
        print(f"  Source ref: {candidate.evidence_ref}")
    print()

In [ ]:
# What happens when the pattern doesn't match?
bad_sample = "Invoice number: 42 (no pattern match possible)"
bad_result = paxman.normalize(bad_sample, Order)
print(f"Status: {bad_result.status.name}")
print(f"Unresolved: {bad_result.unresolved_fields}")

## How it works

1. The **planner** detects `pattern` on the `invoice_number` and `date` fields.
2. It assigns `regex_extraction` before `text_extraction` for these fields.
3. If the regex matches, the matched value is used directly. If not, `text_extraction` may still produce a candidate.
4. The **reconciler** compares candidates from both strategies and picks the most confident.

> **Note:** The `pattern` field follows Python's `re` module syntax. Avoid ReDoS-vulnerable patterns in production (see issue #62).

## Try it yourself

- Add an email field with a regex pattern and test it against different inputs.
- What happens if you provide a pattern that matches multiple times in the input?
- Write a regex for a phone number format and test edge cases.